# Lab 7: Kafka Earthquake Streaming

Data source: USGS Earthquake API

https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_month.geojson

Streaming method: Apache Kafka

Kafka topic: earthquakes

Objective: Receive earthquake messages, process them,
save the dataset in Parquet, and prepare metadata
and a data dictionary.

In [13]:
import json
from pathlib import Path
from uuid import uuid4
from datetime import datetime, timezone
from confluent_kafka import Consumer


In [14]:
consumer = Consumer({
    "bootstrap.servers": "localhost:9092",
    "group.id": f"lab7-{uuid4()}",
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False
})

consumer.subscribe(["earthquakes"])


In [15]:
messages = []
empty_polls = 0

collection_time = datetime.now(timezone.utc).isoformat()


In [16]:
while len(messages) < 100 and empty_polls < 15:

    msg = consumer.poll(1.0)

    if msg is None:
        empty_polls += 1
        continue

    if msg.error():
        print("Kafka error:", msg.error())
        break

    empty_polls = 0

    try:
        record = json.loads(msg.value().decode("utf-8"))
        messages.append(record)

    except (json.JSONDecodeError, UnicodeDecodeError) as error:
        print("Invalid message:", error)


In [17]:
consumer.close()

print("Received messages:", len(messages))
print("Collection time:", collection_time)

Received messages: 100
Collection time: 2026-11-28T03:46:40.730255+00:00


In [18]:
Path("data").mkdir(exist_ok=True)

with open("data/raw_stream.json", "w") as file:
    json.dump(messages, file, indent=2)

print("Raw streaming data saved")

Raw streaming data saved


In [19]:
import pandas as pd

df = pd.DataFrame(messages)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 records:")
display(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate IDs:", df["id"].duplicated().sum())

Dataset shape: (100, 10)

Columns:
['id', 'magnitude', 'place', 'time_ms', 'status', 'event_type', 'longitude', 'latitude', 'depth_km', 'event_time']

First 5 records:


,id,magnitude,place,time_ms,status,event_type,longitude,latitude,depth_km,event_time
0,ci41579904,2.16,"22 km SE of Bodfish, CA",1795538555480,automatic,earthquake,-118.351330,35.431500,2.79,2026-11-24T16:42:35.480Z
1,ci41579896,2.42,"19 km ENE of Little Lake, CA",1795538423350,automatic,earthquake,-117.720830,36.009170,1.71,2026-11-24T16:40:23.350Z
2,aka2026xhgksb,4.20,"11 km SW of Point MacKenzie, Alaska",1795538251142,automatic,earthquake,-150.132000,61.290000,28.20,2026-11-24T16:37:31.142Z
3,hv75075942,1.73,"16 km NNW of Volcano, Hawaii",1795537661900,automatic,earthquake,-155.306671,19.576834,4.09,2026-11-24T16:27:41.900Z
4,hv75075937,2.10,"23 km ENE of Honaunau-Napoopoo, Hawaii",1795537657640,automatic,earthquake,-155.648666,19.501167,6.26,2026-11-24T16:27:37.640Z



Data types:
id                str
magnitude     float64
place             str
time_ms         int64
status            str
event_type        str
longitude     float64
latitude      float64
depth_km      float64
event_time        str
dtype: object

Missing values:
id            0
magnitude     0
place         0
time_ms       0
status        0
event_type    0
longitude     0
latitude      0
depth_km      0
event_time    0
dtype: int64

Duplicate IDs: 0


In [20]:
clean_df = df.copy()


In [21]:
clean_df = clean_df.drop(columns=["batch_id"], errors="ignore")

In [22]:
numeric_columns = [
    "magnitude",
    "longitude",
    "latitude",
    "depth_km",
    "time_ms"
]

In [23]:
for col in numeric_columns:
    clean_df[col] = pd.to_numeric(
        clean_df[col],
        errors="coerce"
    )

In [24]:
clean_df["event_time"] = pd.to_datetime(
    clean_df["event_time"],
    utc=True,
    errors="coerce"
)

In [25]:
clean_df["place"] = clean_df["place"].fillna("Unknown location").str.strip()

clean_df["status"] = (
    clean_df["status"]
    .fillna("unknown")
    .str.strip()
    .str.lower()
)

In [26]:
clean_df = clean_df.dropna(
    subset=["id", "magnitude", "longitude",
            "latitude", "depth_km", "time_ms", "event_time"]
)


In [27]:

clean_df = clean_df[
    clean_df["magnitude"].between(-2, 10) &
    clean_df["latitude"].between(-90, 90) &
    clean_df["longitude"].between(-180, 180) &
    clean_df["depth_km"].between(-10, 800)
]


In [28]:
print("Original records:", len(df))
print("Cleaned records:", len(clean_df))
print("Removed records:", len(df) - len(clean_df))

print("\nCleaned data types:")
print(clean_df.dtypes)

print("\nMissing values:")
print(clean_df.isna().sum())

display(clean_df.head())

Original records: 100
Cleaned records: 100
Removed records: 0

Cleaned data types:
id                            str
magnitude                 float64
place                         str
time_ms                     int64
status                        str
event_type                    str
longitude                 float64
latitude                  float64
depth_km                  float64
event_time    datetime64[us, UTC]
dtype: object

Missing values:
id            0
magnitude     0
place         0
time_ms       0
status        0
event_type    0
longitude     0
latitude      0
depth_km      0
event_time    0
dtype: int64


,id,magnitude,place,time_ms,status,event_type,longitude,latitude,depth_km,event_time
0,ci41579904,2.16,"22 km SE of Bodfish, CA",1795538555480,automatic,earthquake,-118.351330,35.431500,2.79,2026-11-24 16:42:35.480+00:00
1,ci41579896,2.42,"19 km ENE of Little Lake, CA",1795538423350,automatic,earthquake,-117.720830,36.009170,1.71,2026-11-24 16:40:23.350+00:00
2,aka2026xhgksb,4.20,"11 km SW of Point MacKenzie, Alaska",1795538251142,automatic,earthquake,-150.132000,61.290000,28.20,2026-11-24 16:37:31.142+00:00
3,hv75075942,1.73,"16 km NNW of Volcano, Hawaii",1795537661900,automatic,earthquake,-155.306671,19.576834,4.09,2026-11-24 16:27:41.900+00:00
4,hv75075937,2.10,"23 km ENE of Honaunau-Napoopoo, Hawaii",1795537657640,automatic,earthquake,-155.648666,19.501167,6.26,2026-11-24 16:27:37.640+00:00


In [ ]:
Path("data").mkdir(exist_ok=True)


In [ ]:
clean_df = clean_df.reset_index(drop=True)

In [ ]:
clean_df.to_parquet(
    "data/earthquakes_cleaned.parquet",
    engine="pyarrow",
    index=False
)

In [ ]:
stored_df = pd.read_parquet(
    "data/earthquakes_cleaned.parquet"
)

In [ ]:
print("Original cleaned shape:", clean_df.shape)
print("Stored dataset shape:", stored_df.shape)

print("\nStored data types:")
print(stored_df.dtypes)

display(stored_df.head())